In [1]:
import sys
import os

tbl_name = "air_traffic"

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.data import get_ibis_connection


In [2]:
postgres_config = {
    "user": "postgres",
    "password": "password",
    "host": "postgres",
    "port": 5432,
    "database": "my_db",
}

con = get_ibis_connection(
    backend="postgres",
    postgres_config=postgres_config,
)

# csv_path = project_root + "/data/air_traffic_gold.csv"
# con = get_ibis_connection(
#     backend="duckdb",
#     duckdb_csv_path=csv_path,
# )


In [3]:
from sql_ai_agent.db_handler import get_tbl_attr

tbl_attr = get_tbl_attr(con=con, tbl_name=tbl_name)

schema = tbl_attr.schema

In [4]:
from langchain_openai import ChatOpenAI

base_url = "https://api.openai.com/v1"
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-4o"

llm = ChatOpenAI(
  base_url=base_url, 
  api_key=api_key, 
  temperature=0, 
  model=model
  )

In [5]:
from langchain_core.prompts import ChatPromptTemplate

system_template = """
Given the following SQL table, your job is to write queries given a user’s request.
Return just the SQL query as plain text, without additional text, and don't use markdown format. 
Please ensure that the field names in the query are enclosed in double quotes.

{additional_context}

CREATE TABLE {tbl_name} ({schema})

""".strip()

user_template = "Write a SQL query that returns: {question}"

messages = [("system", system_template), ("user", user_template)]

prompt_template = ChatPromptTemplate.from_messages(messages)


In [6]:
chain = prompt_template | llm

In [7]:
def basic_sql_agent(chain, question, tbl_name, schema, con, additional_context=""):
    llm_output = chain.invoke(
        {
            "question": question,
            "tbl_name": tbl_name,
            "schema": schema,
            "additional_context": additional_context,
        }
    )
    query = llm_output.content
    print("The return SQL query:")
    print("_" * 60)
    print(query)
    print("_" * 60)
    output = con.sql(query).execute()
    return output


In [8]:
basic_sql_agent(
    chain,
    question="How many passengers were in transit in 2024?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context="",
    con=con,
)


The return SQL query:
____________________________________________________________
SELECT SUM("Passenger Count") AS "Total Passengers In Transit"
FROM air_traffic
WHERE "Activity Type Code" = 'Transit' AND EXTRACT(YEAR FROM "Date") = 2024;
____________________________________________________________


,Total Passengers In Transit
0,None


In [10]:
con.sql('SELECT DISTINCT "Activity Type Code" FROM air_traffic').execute()

,Activity Type Code
0,Deplaned
1,Thru / Transit
2,Enplaned


In [11]:
basic_sql_agent(
    chain,
    question="How many passengers were in transit in 2024?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context="The 'Activity Type Code' unique values are: 'Deplaned', 'Enplaned', and 'Thru / Transit'",
    con=con,
)


The return SQL query:
____________________________________________________________
SELECT SUM("Passenger Count") AS "Total Passengers In Transit"
FROM air_traffic
WHERE "Activity Type Code" = 'Thru / Transit' AND EXTRACT(YEAR FROM "Date") = 2024;
____________________________________________________________


,Total Passengers In Transit
0,77159


In [12]:
from sql_ai_agent.db_handler import get_character_distinct_values
from sql_ai_agent.prompt_handler import format_distinct_values_for_prompt

distinct_values = get_character_distinct_values(
    con=con, tbl_schema=tbl_attr, tbl_name=tbl_name
)

print(distinct_values)


{'Operating Airline': ['ABC Aerolineas S.A. de C.V. dba Interjet', 'Aer Lingus, Ltd.', 'Aeroflot Russian International Airlines', 'Aeromexico', 'Air 2000', 'Air Atlanta Icelandic', 'Air Berlin', 'Air Canada', 'Air Canada Jazz', 'Air China', 'Air Europe', 'Air France', 'Air India Limited', 'Air Italy S.P.A', 'Air New Zealand', 'Air Pacific Limited dba Fiji Airways', 'Air Premia, Inc.', 'AirTran Airways', 'Air Transat', 'Air Wisconsin', 'Alaska Airlines', 'Alitalia Airlines', 'Allegiant Air', 'Allegro Airlines', 'All Nippon Company Airways, Ltd.', 'American Airlines', 'American Eagle Airlines', 'Ameriflight', 'Asiana Airlines', 'ATA Airlines', 'Atlantic Southeast Airlines', 'Atlas Air, Inc', 'BelAir Airlines', 'Boeing Company', 'Breeze Aviation Group, Inc.', 'British Airways', 'Canadian Airlines', 'Casino Express', 'Cathay Pacific', 'Champion Air', 'China Airlines', 'China Eastern', 'China Eastern Airlines, Inc', 'China Southern', 'Comair', 'Compass Airlines', 'Condor Flugdienst GmbH', '

In [13]:
distinct_values_formatted = format_distinct_values_for_prompt(distinct_values)
print(distinct_values_formatted)


The following columns have known categorical values:
- "Operating Airline": 'ABC Aerolineas S.A. de C.V. dba Interjet', 'Aer Lingus, Ltd.', 'Aeroflot Russian International Airlines', 'Aeromexico', 'Air 2000', 'Air Atlanta Icelandic', 'Air Berlin', 'Air Canada', 'Air Canada Jazz', 'Air China' ...
- "Operating Airline IATA Code": '4O', '4T', '5Y', '9W', 'A8', 'AA', 'AB', 'AC', 'AF', 'AI' ...
- "Published Airline": 'ABC Aerolineas S.A. de C.V. dba Interjet', 'Aer Lingus, Ltd.', 'Aeroflot Russian International Airlines', 'Aeromexico', 'Air 2000', 'Air Atlanta Icelandic', 'Air Berlin', 'Air Canada', 'Air China', 'Air Europe' ...
- "Published Airline IATA Code": '4O', '4T', '5Y', '9W', 'A8', 'AA', 'AB', 'AC', 'AF', 'AI' ...
- "GEO Summary": 'Domestic', 'International'
- "GEO Region": 'Asia', 'Australia / Oceania', 'Canada', 'Central America', 'Europe', 'Mexico', 'Middle East', 'South America', 'US'
- "Activity Type Code": 'Deplaned', 'Enplaned', 'Thru / Transit'
- "Price Category Code": 'Low

In [14]:
basic_sql_agent(
    chain,
    question="How many passengers were in transit in 2024?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context=distinct_values_formatted,
    con=con,
)


The return SQL query:
____________________________________________________________
SELECT SUM("Passenger Count") AS "Total Passengers In Transit"
FROM air_traffic
WHERE "Activity Type Code" = 'Thru / Transit' AND EXTRACT(YEAR FROM "Date") = 2024;
____________________________________________________________


,Total Passengers In Transit
0,77159
